In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# create driver
from src.scraper.driver import make_driver
driver = make_driver(headless=True)

In [ ]:
from src.config import PORTAL_2025, PORTAL_2024
from src.scraper.link_extractor import scrape_portal_player_links


# urls_2024 = scrape_portal_player_links(driver, PORTAL_2024)
# print(f"Found {len(urls_2024)} player transfer portal links for 2024 class.")

urls_2025 = scrape_portal_player_links(driver, PORTAL_2025)
print(f"Found {len(urls_2025)} player transfer portal links for 2025 class.")

with open("data/urls_2025_transfers.json", "w") as f:
    json.dump(urls_2025, f)

# all_urls = list(dict.fromkeys(urls_2025 + urls_2024))
# print("total portal urls:", len(all_urls))

Clicking 'Load More':   0%|          | 0/200 [00:00<?, ?click/s]

Found 3010 player transfer portal links for 2025 class.


In [5]:
# urls_2025[2:]
# first two results erroneous cbssports.com links

In [6]:
## tests
# from src.scraper.player_scraper import scrape_player
# p = scrape_player(driver, 'https://247sports.com/player/emmanuel-pregnon-46140927/college-299861/')
# p

# from src.utils.tests import test_timeline
# events = test_timeline(driver, "https://247sports.com/player/howard-sampson-46129672/college-310950/")
# len(events)

# from src.config import DEBUG
# from src.scraper.player_scraper import scrape_player

# rows = []
# for u in tqdm(urls_2025[2:102]):
#     try:
#         d = scrape_player(driver, u)
#         rows.append(d)
#     except Exception as e:
#         print("FAIL", u, e)

# portal_df = pd.DataFrame(rows)
# portal_df.isna().sum()

In [3]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [ ]:
from src.storage.cache_new import run_scrape
# from src.config import CACHE_PATH

CACHE_PATH = "data/portal_cache_2025_1218_run5.jsonl"

all_urls = list(dict.fromkeys(urls_2025[2:]))
scraped_count, cache_count = run_scrape(
    all_urls,
    out_path=CACHE_PATH,
    num_workers=8,        # try 4, 6, 8
    recycle_every=100     # try 50 if you see instability
)
print("scraped this run:", scraped_count, "already cached:", cache_count)


Scraping:   0%|          | 0/3008 [00:00<?, ?player/s]

KeyboardInterrupt: 

In [ ]:
from src.storage.cache_new import load_cache

test = load_cache(CACHE_PATH)
test_df = pd.DataFrame(test).T.set_index('id_247')
test_clean = {k: v for k, v in test.items() if isinstance(v, dict) and "name" in v}
test_clean_df = pd.DataFrame(test_clean).T.set_index('id_247')
test_clean_df.to_csv(CACHE_PATH[:-5] + 'csv')

with open(CACHE_PATH, "w") as f:
    for v in test.values():
        if isinstance(v, dict) and "name" in v:
            f.write(json.dumps(v) + "\n")

In [30]:
# test_clean_df.to_csv("data/portal_2025_transfers_1218_run4.csv")
# test_clean

with open("data/portal_2025_transfers_1218_run5.jsonl", "w") as f:
    for v in test.values():
        if isinstance(v, dict) and "name" in v:
            f.write(json.dumps(v) + "\n")

## old

In [ ]:
from src.scraper.player_scraper import scrape_player
from src.storage.cache import load_cache, append_cache, scrape_one
from src.config import CACHE_PATH, WORKERS

# Load cache + filter todo
cache = load_cache(CACHE_PATH)
# todo = [u for u in all_urls if u not in cache]
todo = [u for u in urls_2025[2:] if u not in cache]
print("cached:", len(cache), "todo:", len(todo))

# Start with cached rows
rows = list(cache.values())

# Parallel scrape todo with checkpointing
fails = []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(scrape_one, u): u for u in todo}
    for fut in tqdm(as_completed(futs), total=len(futs)):
        u = futs[fut]
        try:
            d = fut.result()
            rows.append(d)
            append_cache(d, CACHE_PATH)
        except Exception as e:
            fails.append((u, str(e)))
            print("FAIL:", u, e)

print("done. scraped:", len(rows), "failed:", len(fails))

portal_df = pd.DataFrame(rows)

# Optional: save a clean CSV snapshot too
# portal_df.to_csv("portal_scrape_2024_2025.csv", index=False)

portal_df.head()

cached: 0 todo: 3008


  0%|          | 0/3008 [00:00<?, ?it/s]

FAIL: https://247sports.com/player/xavier-chaplin-46117935/college-282273 HTTPConnectionPool(host='localhost', port=52371): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/damon-wilson-ii-46114588/college-291463 HTTPConnectionPool(host='localhost', port=50724): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/miquise-humphrey-grace-46086136/college-324656 HTTPConnectionPool(host='localhost', port=50263): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/jerry-wilson-46102475/college-311515 HTTPConnectionPool(host='localhost', port=55884): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/tj-searcy-46112079/college-290478 HTTPConnectionPool(host='localhost', port=55108): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/walker-white-46117707/college-307133 HTTPConnectionPool(host='localhost', port=65518): Read timed out. (read timeout=120)
FAIL: https://247sports.com/player/mark-hamper

In [ ]:
# Optional: save a clean CSV snapshot too
portal_df.to_csv("portal_scrape_2024_2025.csv", index=False)

portal_df.head()

In [9]:
from src.storage.cache import load_cache
from src.config import CACHE_PATH
portal_df_test = pd.DataFrame(load_cache(CACHE_PATH)).T.set_index('id_247')

In [18]:
# portal_df_test[['pos_247', 'hs_pos']].tail(15)
portal_df_test[portal_df_test['hs_pos'].isna()].iloc[0]#['source_player_url']#['source_hs_url']#
# portal_df_test.set_index('name').loc['Nico Iamaleava']#[].head(15)#\
# portal_df_test.isna().sum()
# portal_df_test.reset_index().head(10).loc[5]['source_player_url']
# portal_df_test.head()
# fix pos_247, transfer_pos_rank bug
# rating bug

name                                                     Emmanuel Pregnon
pos_247                                                               IOL
hs_name                                                         Jefferson
hs_city                                                            Denver
hs_state                                                               CO
transfer_rating                                                        93
transfer_year                                                        2025
transfer_ovr_rank                                                      21
transfer_pos_rank                                                       3
transfer_stars                                                          4
transfer_origin                                                   Wyoming
transfer_destination                                               Oregon
hs_class                                                             2020
hs_rating_247                         

In [12]:
portal_df_test

,name,pos_247,hs_name,hs_city,hs_state,transfer_rating,transfer_year,transfer_ovr_rank,transfer_pos_rank,transfer_stars,...,transfer_destination,hs_class,hs_rating_247,hs_pos,composite_rating,composite_natl_rank,composite_pos_rank,source_hs_url,hs_stars,source_player_url
id_247,,,,,,,,,,,,,,,,,,,,,
46078807,John Mateer,QB,Little Elm,Little Elm,TX,95,2025,6,3,4,...,Oklahoma,2022,82,QB,0.8267,1766,119,https://247sports.com/player/john-mateer-46078...,3,https://247sports.com/player/john-mateer-46078...
46100635,Isaiah World,OT,Lincoln,San Diego,CA,98,2025,2,1,5,...,Oregon,2021,81,OT,0.8322,1682,137,https://247sports.com/player/isaiah-world-4610...,3,https://247sports.com/player/isaiah-world-4610...
46134398,Eric Singleton Jr.,WR,Alexander,Douglasville,GA,96,2025,5,1,4,...,Auburn,2023,88,WR,0.8619,1072,149,https://247sports.com/player/eric-singleton-jr...,3,https://247sports.com/player/eric-singleton-jr...
46053141,Carson Beck,QB,Mandarin,Jacksonville,FL,96,2025,4,2,4,...,Miami,2020,92,PRO,0.9095,254,9,https://247sports.com/player/carson-beck-46053...,4,https://247sports.com/player/carson-beck-46053...
46114588,Damon Wilson II,EDGE,Venice,Venice,FL,98,2025,3,1,5,...,Missouri,2023,97,EDGE,0.9869,17,3,https://247sports.com/player/damon-wilson-ii-4...,4,https://247sports.com/player/damon-wilson-ii-4...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46158015,Roy Alexander,WR,Bridgton Academy,Fort Myers,FL,87,2025,374,65,3,...,Texas Tech,2020,None,None,None,None,None,https://247sports.com/player/roy-alexander-461...,None,https://247sports.com/player/roy-alexander-461...
46100208,Brenen Thompson,WR,Spearman,Spearman,TX,87,2025,377,68,3,...,Mississippi State,2022,94,WR,0.9476,131,20,https://247sports.com/player/brenen-thompson-4...,4,https://247sports.com/player/brenen-thompson-4...
46100431,Isaiah Sategna,WR,Fayetteville,Fayetteville,AR,87,2025,376,67,3,...,Oklahoma,2022,94,WR,0.9432,137,23,https://247sports.com/player/isaiah-sategna-46...,4,https://247sports.com/player/isaiah-sategna-46...


In [5]:
driver.quit()